# box-array-to-tensor-with-recipe — faded example 2: Compute the requires_grad gate at the wrapper boundary

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `box-array-to-tensor-with-recipe`. The last cell reports your progress on the `Backprop: Box array as Tensor + recipe` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Box array as Tensor + recipe` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`box-array-to-tensor-with-recipe`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "box-array-to-tensor-with-recipe"
DD_SUBTOPIC = "Backprop: Box array as Tensor + recipe"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The gate that decides whether to track is **global grad-tracking AND any input requiring grad**. Both signals combine with AND: dropping the global toggle lets a no_grad block silently build a graph; dropping the any-input check builds useless Recipes for constant-only calls. The gate result drives both the boxed tensor's flag and whether a Recipe is attached.

## Faded exercise 2

`box_mul` wraps a binary multiply. Everything is given except the gate. **Complete the line that computes `requires_grad`** as the AND of the global toggle `grad_tracking_enabled` and whether any positional arg is a MiniTensor with `requires_grad` True.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
from typing import Callable
from dataclasses import dataclass

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array)
        self.requires_grad = requires_grad
        self.recipe = None

@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict

def multiply(a, b):
    return a * b

def box_mul(args, grad_tracking_enabled=True):
    raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
    requires_grad = None  # TODO: fill in this step — read the prompt cell above
    out_raw = multiply(*raw_args)
    parents = {}
    if requires_grad:
        parents = {idx: a for idx, a in enumerate(args)
                   if isinstance(a, MiniTensor) and a.requires_grad}
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    if requires_grad:
        out.recipe = Recipe(multiply, raw_args, {}, parents)
    return out


def _test():
    a = MiniTensor(np.array([2.0, 3.0]), requires_grad=True)
    b = MiniTensor(np.array([4.0, 5.0]), requires_grad=False)
    out = box_mul((a, b), grad_tracking_enabled=True)
    assert np.allclose(out.array, a.array * b.array)
    assert out.requires_grad is True
    assert out.recipe is not None
    assert out.recipe.parents == {0: a}
    out_off = box_mul((a, b), grad_tracking_enabled=False)
    assert out_off.requires_grad is False
    assert out_off.recipe is None
    c = MiniTensor(np.array([1.0]), requires_grad=False)
    out_const = box_mul((c, c), grad_tracking_enabled=True)
    assert out_const.requires_grad is False
    assert out_const.recipe is None


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from typing import Callable
from dataclasses import dataclass

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array)
        self.requires_grad = requires_grad
        self.recipe = None

@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict

def multiply(a, b):
    return a * b

def box_mul(args, grad_tracking_enabled=True):
    raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
    requires_grad = grad_tracking_enabled and any(
        isinstance(a, MiniTensor) and a.requires_grad for a in args
    )
    out_raw = multiply(*raw_args)
    parents = {}
    if requires_grad:
        parents = {idx: a for idx, a in enumerate(args)
                   if isinstance(a, MiniTensor) and a.requires_grad}
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    if requires_grad:
        out.recipe = Recipe(multiply, raw_args, {}, parents)
    return out
```
</details>